In [5]:
%reset -f

In [3]:
import torch

print(torch.__version__)

2.2.0+cu121


In [4]:
import pandas as pd


all = pd.read_csv("./data/all_gene_disease_associations.tsv", sep="\t")
bio = pd.read_csv("./data/BIOGRID-ORGANISM-Homo_sapiens-4.4.231.tab3.txt", sep="\t")
dis = pd.read_csv("./data/disease_associations.tsv", sep="\t")
gene = pd.read_csv("./data/gene_associations.tsv", sep="\t")

scores = pd.read_csv("/raid/anurag/pytorch_codes/gene_disease/data/APU_scores/C0006142_Malignant_neoplasm_of_breast_features_Score.csv")


/tmp/ipykernel_816899/4292139961.py:5: DtypeWarning: Columns (1,2,18) have mixed types. Specify dtype option on import or set low_memory=False.
  bio = pd.read_csv("./data/BIOGRID-ORGANISM-Homo_sapiens-4.4.231.tab3.txt", sep="\t")


In [8]:
#all.info()
all.columns
#all.head(2)

Index(['geneId', 'geneSymbol', 'DSI', 'DPI', 'diseaseId', 'diseaseName',
       'diseaseType', 'diseaseClass', 'diseaseSemanticType', 'score', 'EI',
       'YearInitial', 'YearFinal', 'NofPmids', 'NofSnps', 'source'],
      dtype='object')

In [9]:
#dis.info()
dis.columns
#dis.head(3)

Index(['diseaseId', 'diseaseName', 'diseaseType', 'diseaseClass',
       'diseaseSemanticType', 'NofGenes', 'NofPmids'],
      dtype='object')

In [10]:
#gene.info()
gene.columns
#gene.head(3)

Index(['geneId', 'geneSymbol', 'DSI', 'DPI', 'PLI', 'protein_class_name',
       'protein_class', 'NofDiseases', 'NofPmids'],
      dtype='object')

In [ ]:
bio.info()
bio.head(2)

In [17]:
import sys
from pathlib import Path
projpath = Path('/raid/anurag/pytorch_codes/gene_disease')
sys.path.append(str(projpath / "scripts"))

import json
from scripts.utils.databaseElements import SQLiteHandler


with open(projpath / "scripts/configFiles/preproc_config.json", "r") as cfg:
    config = json.load(cfg)
    #print(config)

    dbObj = SQLiteHandler()

    dbObj.connectionManager(projpath / "data" / config["dbname"])
    exist = dbObj.sqlQuery(config["existViewDiseaseQ"])
    print(exist)
    if len(exist) == 1:
        dbObj.sqlQuery(config["dropViewDiseaseQ"])
    
    dbObj.sqlQuery(config["makeViewDiseaseQ"])

    results = dbObj.sqlQuery(config["geneDiseaseEquivQ"])

    resultsDF = pd.DataFrame(results, columns=[
        "geneID", "geneName", "geneDescription", "DSI", "DPI", "diseaseId",
        "diseaseName", "diseaseType", "diseaseClass", "diseaseClassName",
        "source", "association", "associationType", "pmid", "score", "EL", "EI",
        "year"])

    del dbObj



executed __init__
executed connMgr
executed sqlQ
SELECT name FROM sqlite_master WHERE type="view" AND name="DISEASE"
[('DISEASE',)]
executed sqlQ
DROP VIEW DISEASE
executed sqlQ
CREATE VIEW DISEASE AS SELECT diseaseAttributes.diseaseNID, diseaseAttributes.diseaseID, diseaseAttributes.diseaseName, diseaseAttributes.type as diseaseType, diseaseClass.diseaseClass, diseaseClass.diseaseClassName FROM diseaseAttributes JOIN disease2class ON diseaseAttributes.diseaseNID = disease2class.diseaseNID JOIN diseaseClass ON disease2class.diseaseClassNID = diseaseClass.diseaseClassNID
executed sqlQ
SELECT geneAttributes.geneID, geneAttributes.geneName, geneAttributes.geneDescription, geneAttributes.DSI, geneAttributes.DPI, DISEASE.diseaseId, DISEASE.diseaseName, DISEASE.diseaseType, DISEASE.diseaseClass, DISEASE.diseaseClassName, geneDiseaseNetwork.source, geneDiseaseNetwork.association, geneDiseaseNetwork.associationType, geneDiseaseNetwork.pmid, geneDiseaseNetwork.score, geneDiseaseNetwork.EL, gene

In [19]:
#resultsDF.head(2)
resultsDF.columns

Index(['geneID', 'geneName', 'geneDescription', 'DSI', 'DPI', 'diseaseId',
       'diseaseName', 'diseaseType', 'diseaseClass', 'diseaseClassName',
       'source', 'association', 'associationType', 'pmid', 'score', 'EL', 'EI',
       'year'],
      dtype='object')

In [22]:
all.head(3)

,geneId,geneSymbol,DSI,DPI,diseaseId,diseaseName,diseaseType,diseaseClass,diseaseSemanticType,score,EI,YearInitial,YearFinal,NofPmids,NofSnps,source
0,1,A1BG,0.7,0.538,C0001418,Adenocarcinoma,group,C04,Neoplastic Process,0.01,1.0,2008.0,2008.0,1,0,LHGDN
1,1,A1BG,0.7,0.538,C0002736,Amyotrophic Lateral Sclerosis,disease,C18;C10,Disease or Syndrome,0.01,1.0,2008.0,2008.0,1,0,BEFREE
2,1,A1BG,0.7,0.538,C0003578,Apnea,phenotype,C23;C08,Sign or Symptom,0.01,1.0,2017.0,2017.0,1,0,BEFREE


In [21]:
resultsDF.head(3)

,geneID,geneName,geneDescription,DSI,DPI,diseaseId,diseaseName,diseaseType,diseaseClass,diseaseClassName,source,association,associationType,pmid,score,EL,EI,year
0,1048,CEACAM5,CEA cell adhesion molecule 5,0.420,0.846,C0027651,Neoplasms,group,C04,Neoplasms,BEFREE,None,Biomarker,1000501,0.10,None,0.956175,1976
1,1026,CDKN1A,cyclin dependent kinase inhibitor 1A,0.403,0.769,C0006826,Malignant Neoplasms,group,C04,Neoplasms,BEFREE,None,GeneticVariation,10021299,0.40,None,0.987013,1999
2,1026,CDKN1A,cyclin dependent kinase inhibitor 1A,0.403,0.769,C0007103,Malignant neoplasm of endometrium,disease,C04,Neoplasms,BEFREE,None,GeneticVariation,10021299,0.02,None,1.000000,1999


In [1]:
import sys
from pathlib import Path
projpath = Path('/raid/anurag/pytorch_codes/gene_disease')
sys.path.append(str(projpath / "scripts"))
from scripts.preprocess import DataGraph


cfgFile = Path("/raid/anurag/pytorch_codes/gene_disease/scripts/configFiles/preproc_config.json")

grphObj = DataGraph(str(cfgFile))
#grphObj.buildGraph()
grphObj.prepareFeatures()


reading in the PPI (BioGRID) graph...done!
reading in the disease data...done !
computing features...
degree count 19761
 ring count 19761
 NetRank count 19761
 NetShort count 19761
 HeatDiff count 19761
 InfoDiff count 19761

done !
adding APU scores to the graph...done !
saving graph to disk with features...done !


In [5]:
#bio.info()
#bio.columns
#bio.dtypes
bio.head(5)

,#BioGRID Interaction ID,Entrez Gene Interactor A,Entrez Gene Interactor B,BioGRID ID Interactor A,BioGRID ID Interactor B,Systematic Name Interactor A,Systematic Name Interactor B,Official Symbol Interactor A,Official Symbol Interactor B,Synonyms Interactor A,...,TREMBL Accessions Interactor B,REFSEQ Accessions Interactor B,Ontology Term IDs,Ontology Term Names,Ontology Term Categories,Ontology Term Qualifier IDs,Ontology Term Qualifier Names,Ontology Term Types,Organism Name Interactor A,Organism Name Interactor B
0,103,6416,2318,112315,108607,-,-,MAP2K4,FLNC,JNKK|JNKK1|MAPKK4|MEK4|MKK4|PRKMK4|SAPKK-1|SAP...,...,Q59H94,NP_001120959|NP_001449,-,-,-,-,-,-,Homo sapiens,Homo sapiens
1,117,84665,88,124185,106603,-,-,MYPN,ACTN2,CMD1DD|CMH22|MYOP|RCM4,...,Q59FD9|F6THM6,NP_001094|NP_001265272|NP_001265273,-,-,-,-,-,-,Homo sapiens,Homo sapiens
2,183,90,2339,106605,108625,-,-,ACVR1,FNTA,ACTRI|ACVR1A|ACVRLK2|ALK2|FOP|SKR1|TSRI,...,-,NP_002018,-,-,-,-,-,-,Homo sapiens,Homo sapiens
3,278,2624,5371,108894,111384,-,-,GATA2,PML,DCML|IMD21|MONOMAC|NFE1B,...,-,NP_150250|NP_150253|NP_150252|NP_150247|NP_150...,-,-,-,-,-,-,Homo sapiens,Homo sapiens
4,418,6118,6774,112038,112651,RP4-547C9.3,-,RPA2,STAT3,REPA2|RP-A p32|RP-A p34|RPA32,...,-,NP_644805|NP_003141|NP_001356447|NP_001356443|...,-,-,-,-,-,-,Homo sapiens,Homo sapiens


In [1]:
#scores.head(5)
scores.info()

NameError: name 'scores' is not defined

In [7]:
%reset -f

import pandas as pd

ranking = {
    "abc": [4, 0.1],
    "bcd": [1, 0.002],
    "cde": [2, 0.08],
    "def": [0, 0.2],
    "efg": [3, 0.001],
    "fgh": [1, 0.05],
    "ghi": [2, 0.03],
}

df = pd.DataFrame.from_dict(
    ranking,
    orient="index",
    columns=["Pseudo-labels", "Score"]
)
df["GeneID"] = df.index
df.reset_index(drop=True, inplace=True)
df = df[["GeneID", "Score", "Pseudo-labels"]]

df.sort_values(
    ["Score", "Pseudo-labels"],
    ascending=[False, False],
    inplace=True,
    ignore_index=True
)

print(df)

  GeneID  Score  Pseudo-labels
0    def  0.200              0
1    abc  0.100              4
2    cde  0.080              2
3    fgh  0.050              1
4    ghi  0.030              2
5    bcd  0.002              1
6    efg  0.001              3


In [12]:
import torch

tnsr = torch.Tensor([[3], [1], [10], [9], [5], [2], [-6], [-43]])
print(tnsr)
print(tnsr.shape)
del_pos = [2,5,7]

indices_to_keep = [i for i in range(tnsr.shape[0]) if i not in del_pos]
new_data = tnsr[torch.LongTensor(indices_to_keep)]
output = torch.index_select(tnsr, 0, torch.LongTensor(indices_to_keep))

print(new_data)
print(new_data.shape)
print(output)
print(new_data.shape)

tensor([[  3.],
        [  1.],
        [ 10.],
        [  9.],
        [  5.],
        [  2.],
        [ -6.],
        [-43.]])
torch.Size([8, 1])
tensor([[ 3.],
        [ 1.],
        [ 9.],
        [ 5.],
        [-6.]])
torch.Size([5, 1])
tensor([[ 3.],
        [ 1.],
        [ 9.],
        [ 5.],
        [-6.]])
torch.Size([5, 1])


In [1]:
import networkx as nx

g1 = nx.read_gml('/raid/anurag/pytorch_codes/gene_disease/data/DataGraphs/grafo_nedbit_C0006142.gml')

g2 = nx.read_gml('/raid/anurag/pytorch_codes/gene_disease/data/Graphs/C0006142_nedbit.gml')

In [9]:
print('Train input graph: No. of nodes- {}, No. of edges- {}'\
    .format(g2.number_of_edges(), g2.number_of_nodes()))

print('Ranking input graph: No. of nodes- {}, No. of edges- {}'\
    .format(g1.number_of_edges(), g1.number_of_nodes()))

# print('Are Python objects same: {}'\
#     .format(nx.graphs_equal(g1, g2)))

Train input graph: No. of nodes- 678932, No. of edges- 19761
Ranking input graph: No. of nodes- 678932, No. of edges- 19761


In [10]:
diffG = nx.difference(g1, g2)

print('Difference graph: No. of nodes- {}, No. of edges- {}'\
    .format(diffG.number_of_edges(), diffG.number_of_nodes()))

Difference graph: No. of nodes- 0, No. of edges- 19761


In [7]:
print('Could the graphs be isomorphic 1: {}'\
    .format(nx.faster_could_be_isomorphic(g1, g2)))
print('Could the graphs be isomorphic 2: {}'\
    .format(nx.fast_could_be_isomorphic(g1, g2)))
print('Could the graphs be isomorphic 3: {}'\
    .format(nx.could_be_isomorphic(g1, g2)))

Are the graphs isomorphic 1: True
Are the graphs isomorphic 2: True


KeyboardInterrupt: 